# LeetCode 3210 — Find the Encrypted String bằng Python

Link đề: https://leetcode.com/problems/find-the-encrypted-string/ · Độ khó: **Easy** · Chủ đề: String (mô phỏng, dịch vòng)

Skeleton LeetCode:

```python
class Solution:
    def getEncryptedString(self, s: str, k: int) -> str:
        ...
```

Ghi chú: đề và các ví dụ được viết theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: với **mỗi** ký tự `c` của chuỗi `s`, thay `c` bằng **ký tự đứng sau `c` đúng `k` vị trí** trong chuỗi, đi **theo vòng tròn** (hết chuỗi thì quay về đầu); trả về chuỗi sau khi thay.
2. Nắm các công cụ Python: chỉ số chuỗi `s[i]`, phép chia dư `%` để "quấn vòng", cắt lát `s[a:]` / `s[:a]` và nối chuỗi, `"".join(...)`, nhân đôi chuỗi `s + s`, và `collections.deque.rotate`.
3. Đi từ **bước từng ký tự `k` lần** → **rút gọn `k` bằng `k % n`** → **cắt lát và nối (xoay chuỗi)** → **nhân đôi chuỗi rồi lấy đoạn** → **`deque.rotate`**.
4. Hiểu mấu chốt: dịch vòng `n` bước thì về đúng chỗ cũ nên **chỉ cần `k % n` bước**; và vì mọi ký tự cùng dịch **cùng một khoảng**, kết quả chính là **chuỗi `s` xoay trái `k % n` vị trí**.
5. Tránh các lỗi kinh điển: đọc `s[i + k]` **không có `%`** (`IndexError`), sửa chuỗi **tại chỗ** rồi đọc lại phần đã sửa, đi **sai chiều** (lùi thay vì tiến), nhầm "ký tự sau `k` vị trí **trong chuỗi**" với "dịch **bảng chữ cái**" kiểu Caesar, và cắt `s[k:] + s[:k]` khi `k > n`.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và số nguyên `k`. **Mã hóa** chuỗi theo thuật toán sau:

- Với **mỗi** ký tự `c` trong `s`, thay `c` bằng ký tự đứng **sau `c` đúng `k` vị trí** trong chuỗi `s` (theo kiểu **vòng tròn**: sau ký tự cuối là ký tự đầu).

Hãy trả về **chuỗi đã mã hóa**.

**Ví dụ 1**

- Input: `s = "dart"`, `k = 3`
- Output: `"tdar"`
- Giải thích: `i = 0`, ký tự thứ 3 sau `'d'` là `'t'`; `i = 1`, ký tự thứ 3 sau `'a'` là `'d'` (đi `r`, `t`, rồi quay về `d`); `i = 2`, sau `'r'` là `'a'`; `i = 3`, sau `'t'` là `'r'`. Ghép lại `"tdar"`.

**Ví dụ 2**

- Input: `s = "aaa"`, `k = 1`
- Output: `"aaa"`
- Giải thích: mọi ký tự giống nhau nên chuỗi mã hóa giống hệt.

**Ràng buộc:** `1 <= s.length <= 100`; `1 <= k <= 10^4`; `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
import random
import string
from collections import deque

examples = [
    ("dart", 3, "tdar"),
    ("aaa", 1, "aaa"),
]
for s, k, expected in examples:
    print(f"s = {s!r}, k = {k} -> kỳ vọng {expected!r}")

s = 'dart', k = 3 -> kỳ vọng 'tdar'
s = 'aaa', k = 1 -> kỳ vọng 'aaa'


## 3. Kiến thức nền

### 3.1. Chỉ số chuỗi và chuỗi là bất biến

`s[i]` là ký tự ở vị trí `i` (đếm từ `0`). Chuỗi Python là **bất biến** (immutable): không thể gán `s[0] = "x"`. Muốn "sửa" ta phải tạo chuỗi mới, ví dụ gom từng ký tự vào một **list** rồi nối bằng `"".join(list)`.

In [2]:
s = "dart"
print(s[0], s[1], s[3], s[-1])           # d a t t   (s[-1] là ký tự cuối)
try:
    s[0] = "x"
except TypeError as e:
    print("TypeError:", e)
chars = list(s)                          # ['d', 'a', 'r', 't']
chars[0] = "x"
print("".join(chars))                    # xart

d a t t
TypeError: 'str' object does not support item assignment
xart


### 3.2. "Quấn vòng" bằng phép chia dư `%`

Chuỗi `n` ký tự có chỉ số `0..n-1`. Đi tới `n` thì phải quay về `0`: dùng `% n`. Ví dụ `n = 4`: `4 % 4 = 0`, `5 % 4 = 1`, `6 % 4 = 2`. Vậy **ký tự đứng sau ký tự ở vị trí `i` đúng `k` bước** nằm ở vị trí `(i + k) % n`.

In [3]:
n = 4
for x in [0, 1, 2, 3, 4, 5, 6, 7, 8]:
    print(f"{x} % {n} = {x % n}")
s = "dart"
print([ (i + 3) % n for i in range(n) ])       # [3, 0, 1, 2]
print("".join(s[(i + 3) % n] for i in range(n)))      # tdar

0 % 4 = 0
1 % 4 = 1
2 % 4 = 2
3 % 4 = 3
4 % 4 = 0
5 % 4 = 1
6 % 4 = 2
7 % 4 = 3
8 % 4 = 0
[3, 0, 1, 2]
tdar


### 3.3. Vì sao chỉ cần `k % n`?

Đi `n` bước trên vòng tròn `n` ký tự thì trở về **đúng chỗ cũ**. Vậy đi `k` bước và đi `k % n` bước cho cùng kết quả. Với `k` lớn (tới `10^4`) mà `n` chỉ vài chục, việc bớt đi mọi vòng đầy giúp tiết kiệm công sức.

In [4]:
s, n = "abc", 3
for k in [1, 2, 3, 4, 5, 6, 7, 10000]:
    print(f"k={k:>5}: k % n = {k % n}, ký tự sau 'a' (i=0) {k} bước = {s[(0 + k) % n]}")

k=    1: k % n = 1, ký tự sau 'a' (i=0) 1 bước = b
k=    2: k % n = 2, ký tự sau 'a' (i=0) 2 bước = c
k=    3: k % n = 0, ký tự sau 'a' (i=0) 3 bước = a
k=    4: k % n = 1, ký tự sau 'a' (i=0) 4 bước = b
k=    5: k % n = 2, ký tự sau 'a' (i=0) 5 bước = c
k=    6: k % n = 0, ký tự sau 'a' (i=0) 6 bước = a
k=    7: k % n = 1, ký tự sau 'a' (i=0) 7 bước = b
k=10000: k % n = 1, ký tự sau 'a' (i=0) 10000 bước = b


### 3.4. Cắt lát `s[a:]`, `s[:a]` và nối chuỗi

`s[a:]` lấy từ vị trí `a` đến hết, `s[:a]` lấy `a` ký tự đầu, và `+` nối hai chuỗi. Ví dụ `"dart"[3:] = "t"`, `"dart"[:3] = "dar"`, và `"dart"[3:] + "dart"[:3] = "tdar"` — **trùng** đáp án ví dụ 1. Đây là phép **xoay trái** chuỗi.

In [5]:
s = "dart"
print(repr(s[3:]), repr(s[:3]), repr(s[3:] + s[:3]))      # 't' 'dar' 'tdar'
print(repr(s[0:]), repr(s[:0]), repr(s[0:] + s[:0]))      # xoay 0: không đổi
print(repr(s[4:]), repr(s[:4]))                            # '' 'dart': cắt tại độ dài n cho chuỗi rỗng

't' 'dar' 'tdar'
'dart' '' 'dart'
'' 'dart'


### 3.5. Nhân đôi chuỗi `s + s` và `deque.rotate`

`s + s` chứa mọi phép xoay của `s` dưới dạng một **đoạn liên tiếp** dài `n`: xoay trái `r` vị trí chính là `(s + s)[r : r + n]`. Ngoài ra `collections.deque` có phương thức `rotate(m)` xoay **phải** `m` bước; xoay trái `r` bước là `rotate(-r)`.

In [6]:
s = "dart"
d = s + s
print(d, [d[r:r + 4] for r in range(5)])          # 'dartdart' ['dart', 'artd', 'rtda', 'tdar', 'dart']
dq = deque(s)
dq.rotate(-3)                                      # xoay trái 3
print("".join(dq))                                 # tdar

dartdart ['dart', 'artd', 'rtda', 'tdar', 'dart']
tdar


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `s = "dart"`, `k = 3` (`n = 4`): mỗi ký tự "đi" `3` bước về phía **sau** theo vòng tròn.

| `i` | ký tự | đi 3 bước | vị trí tới | ký tự tới |
|---|---|---|---|---|
| 0 | d | 0 → 1 → 2 → 3 | 3 | t |
| 1 | a | 1 → 2 → 3 → 0 | 0 | d |
| 2 | r | 2 → 3 → 0 → 1 | 1 | a |
| 3 | t | 3 → 0 → 1 → 2 | 2 | r |

Ghép các ký tự tới: `t, d, a, r` ⇒ `"tdar"`. Nhìn kỹ: **mọi ký tự dịch cùng một khoảng**, nên kết quả chính là cả chuỗi **xoay** đi một đoạn (ở đây `"dart"` xoay trái 3 vị trí). Điều này gợi ý cách làm một dòng bằng cắt lát.

In [7]:
s, k = "dart", 3
n = len(s)
result = ""
for i in range(n):
    path = [(i + step) % n for step in range(k + 1)]
    print(f"i={i} ({s[i]}): đường đi {path} -> vị trí {path[-1]} -> ký tự {s[path[-1]]}")
    result += s[path[-1]]
print(result)

i=0 (d): đường đi [0, 1, 2, 3] -> vị trí 3 -> ký tự t
i=1 (a): đường đi [1, 2, 3, 0] -> vị trí 0 -> ký tự d
i=2 (r): đường đi [2, 3, 0, 1] -> vị trí 1 -> ký tự a
i=3 (t): đường đi [3, 0, 1, 2] -> vị trí 2 -> ký tự r
tdar


## 5. Cách 1 — Brute Force / baseline: bước từng ký tự `k` lần

**Ý tưởng.** Với mỗi vị trí `i`, đặt `j = i` rồi **lặp `k` lần**: `j = (j + 1) % n` (nhích một vị trí, quay vòng khi hết chuỗi). Sau `k` bước, `s[j]` là ký tự thay thế. Làm đúng theo lời đề, không "thông minh" gì thêm.

In [8]:
def encrypt_step_by_step(s, k):
    n = len(s)
    result = []
    for i in range(n):
        j = i
        for _ in range(k):              # nhích k lần, mỗi lần một vị trí
            j = (j + 1) % n
        result.append(s[j])
    return "".join(result)


for s, k, expected in examples:
    print(repr(encrypt_step_by_step(s, k)), "(kỳ vọng", repr(expected), ")")

'tdar' (kỳ vọng 'tdar' )
'aaa' (kỳ vọng 'aaa' )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Mỗi ký tự đi `k` bước ⇒ **Time `O(n · k)`** (tối đa `100 · 10^4 = 10^6` phép `% n`), **Space `O(n)`** cho kết quả.

**Nút thắt:** ta đi `k` bước dù phần lớn là các vòng đầy vô ích (`k` có thể gấp trăm lần `n`). Chỉ cần `k % n` bước, thậm chí nhảy thẳng tới `(i + k) % n`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — Nhảy thẳng.** Ký tự đứng sau `s[i]` đúng `k` bước ở vị trí `(i + k) % n`: một phép cộng và một phép chia dư thay cho `k` lần nhích. Không cần rút gọn riêng `k % n` vì `(i + k) % n` đã làm điều đó.

**Bước 2 — Nhận ra đó là phép xoay.** Đặt `r = k % n`. Với mọi `i`, `(i + k) % n = (i + r) % n`. Ký tự mới ở vị trí `i` là `s[(i + r) % n]`: với `i + r < n` chính là `s[i + r]` (phần `s[r:]`), còn với `i + r >= n` quay về đầu (`s[:r]`). Nên **kết quả `= s[r:] + s[:r]`**.

Cả hai cách đều `O(n)`.

In [9]:
s, k = "dart", 3
n = len(s)
r = k % n
print("r =", r)
print("".join(s[(i + k) % n] for i in range(n)))      # tdar (nhảy thẳng)
print(s[r:] + s[:r])                                   # tdar (xoay trái r)

r = 3
tdar
tdar


## 7. Các cách tiếp theo

### Cách 2 — Nhảy thẳng tới `(i + k) % n`

**Giải thích từng đoạn.**

1. `n = len(s)`.
2. `for i in range(n)`: với mỗi vị trí `i`, ký tự mới là `s[(i + k) % n]`.
3. Gom vào list rồi nối bằng `"".join`.

Đây là cách **gần đề nhất**, không cần nghĩ tới "xoay".

In [10]:
def encrypt_jump(s, k):
    n = len(s)
    cs = []
    for i in range(n):
        cs.append(s[(i + k) % n])          # nhảy thẳng tới ký tự sau i đúng k bước
    return "".join(cs)


for s, k, expected in examples:
    print(repr(encrypt_jump(s, k)), "(kỳ vọng", repr(expected), ")")

'tdar' (kỳ vọng 'tdar' )
'aaa' (kỳ vọng 'aaa' )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionJump:
    def getEncryptedString(self, s: str, k: int) -> str:
        n = len(s)
        return "".join(s[(i + k) % n] for i in range(n))


print(SolutionJump().getEncryptedString("dart", 3))      # tdar

tdar


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`**.

### Cách 3 — Xoay chuỗi bằng cắt lát (cách nên nộp)

**Giải thích từng đoạn.**

1. `r = k % n`: số vị trí thực sự cần xoay (`0 <= r < n`).
2. `s[r:]` là phần từ vị trí `r` đến hết; `s[:r]` là `r` ký tự đầu.
3. Nối `s[r:] + s[:r]` là chuỗi xoay trái `r` vị trí, đúng với mã hóa.

**Vì sao cần `k % n`?** Nếu `k >= n` mà viết `s[k:] + s[:k]` thì `s[k:]` rỗng và `s[:k]` là cả chuỗi ⇒ trả về `s` — **sai**.

In [12]:
def encrypt_slice(s, k):
    r = k % len(s)
    return s[r:] + s[:r]


for s, k, expected in examples:
    print(repr(encrypt_slice(s, k)), "(kỳ vọng", repr(expected), ")")
print(encrypt_slice("abc", 4), encrypt_slice("abc", 10000))      # bca bca

'tdar' (kỳ vọng 'tdar' )
'aaa' (kỳ vọng 'aaa' )
bca bca


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionSlice:
    def getEncryptedString(self, s: str, k: int) -> str:
        r = k % len(s)
        return s[r:] + s[:r]


print(SolutionSlice().getEncryptedString("aaa", 1))      # aaa

aaa


**Độ phức tạp Cách 3:** **Time `O(n)`** (cắt lát và nối cùng tuyến tính), **Space `O(n)`**.

### Cách 4 — Nhân đôi chuỗi rồi lấy đoạn dài `n`

**Giải thích từng đoạn.**

1. `d = s + s`: chuỗi dài `2n`, chứa mọi phép xoay của `s`.
2. Xoay trái `r` vị trí là đoạn `d[r : r + n]` với `r = k % n`.

Ưu điểm: **không cần ghép hai nửa** bằng tay; nhược điểm: tốn thêm bộ nhớ `2n`.

In [14]:
def encrypt_doubled(s, k):
    n = len(s)
    r = k % n
    d = s + s
    return d[r:r + n]


for s, k, expected in examples:
    print(repr(encrypt_doubled(s, k)), "(kỳ vọng", repr(expected), ")")

'tdar' (kỳ vọng 'tdar' )
'aaa' (kỳ vọng 'aaa' )


In [15]:
# Code nộp LeetCode cho Cách 4
class SolutionDoubled:
    def getEncryptedString(self, s: str, k: int) -> str:
        n = len(s)
        r = k % n
        return (s + s)[r:r + n]


print(SolutionDoubled().getEncryptedString("dart", 3))      # tdar

tdar


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (chuỗi nhân đôi dài `2n`).

### Cách 5 — `collections.deque.rotate`

**Giải thích từng đoạn.**

1. `dq = deque(s)` đưa chuỗi vào hàng đợi hai đầu.
2. `dq.rotate(-k)` xoay **trái** `k` bước (`rotate(m)` dương là xoay phải, âm là xoay trái; nó tự lấy `m` theo vòng nên `k` lớn vẫn đúng).
3. `"".join(dq)` ghép lại thành chuỗi.

In [16]:
def encrypt_deque(s, k):
    dq = deque(s)
    dq.rotate(-k)                  # xoay trái k bước
    return "".join(dq)


for s, k, expected in examples:
    print(repr(encrypt_deque(s, k)), "(kỳ vọng", repr(expected), ")")
print(encrypt_deque("abc", 10000))      # bca

'tdar' (kỳ vọng 'tdar' )
'aaa' (kỳ vọng 'aaa' )
bca


In [17]:
# Code nộp LeetCode cho Cách 5
from collections import deque


class SolutionDeque:
    def getEncryptedString(self, s: str, k: int) -> str:
        dq = deque(s)
        dq.rotate(-k)
        return "".join(dq)


print(SolutionDeque().getEncryptedString("dart", 3))      # tdar

tdar


**Độ phức tạp Cách 5:** **Time `O(n)`** (việc `rotate` tự rút gọn `k` theo độ dài `n`), **Space `O(n)`**.

## 8. TRACE từng bước

Ta trace **Cách 2** (nhảy thẳng) trên `s = "dart"`, `k = 3`, rồi **Cách 3** (cắt lát) trên `s = "abc"`, `k = 10000`.

| `i` | `s[i]` | `i + k` | `(i + k) % n` (`n = 4`) | ký tự mới | `cs` sau bước |
|---|---|---|---|---|---|
| 0 | d | 3 | 3 | t | `['t']` |
| 1 | a | 4 | 0 | d | `['t', 'd']` |
| 2 | r | 5 | 1 | a | `['t', 'd', 'a']` |
| 3 | t | 6 | 2 | r | `['t', 'd', 'a', 'r']` |

Kết quả `"tdar"`. Với `s = "abc"`, `k = 10000`: `n = 3`, `r = 10000 % 3 = 1` (vì `10000 = 3 · 3333 + 1`), `s[1:] = "bc"`, `s[:1] = "a"` ⇒ `"bca"`.

In [18]:
def trace_jump(s, k):
    n = len(s)
    print(f"s = {s!r}, k = {k}, n = {n}")
    cs = []
    for i in range(n):
        j = (i + k) % n
        cs.append(s[j])
        print(f"i={i}, s[i]={s[i]!r}, i+k={i + k}, (i+k)%n={j}, ký tự mới={s[j]!r}, cs={cs}")
    return "".join(cs)


assert trace_jump("dart", 3) == "tdar"

s = 'dart', k = 3, n = 4
i=0, s[i]='d', i+k=3, (i+k)%n=3, ký tự mới='t', cs=['t']
i=1, s[i]='a', i+k=4, (i+k)%n=0, ký tự mới='d', cs=['t', 'd']
i=2, s[i]='r', i+k=5, (i+k)%n=1, ký tự mới='a', cs=['t', 'd', 'a']
i=3, s[i]='t', i+k=6, (i+k)%n=2, ký tự mới='r', cs=['t', 'd', 'a', 'r']


In [19]:
def trace_slice(s, k):
    n = len(s)
    r = k % n
    print(f"s = {s!r}, k = {k}, n = {n}, r = k % n = {r}")
    print(f"s[r:] = {s[r:]!r}, s[:r] = {s[:r]!r}")
    result = s[r:] + s[:r]
    print("kết quả =", repr(result))
    return result


assert trace_slice("abc", 10000) == "bca"
assert trace_slice("dart", 3) == "tdar"

s = 'abc', k = 10000, n = 3, r = k % n = 1
s[r:] = 'bc', s[:r] = 'a'
kết quả = 'bca'
s = 'dart', k = 3, n = 4, r = k % n = 3
s[r:] = 't', s[:r] = 'dar'
kết quả = 'tdar'


In [20]:
def trace_one_char_steps(s, k, i):
    n = len(s)
    j = i
    print(f"ký tự thứ {i} ({s[i]!r}) đi {k} bước:")
    for step in range(1, k + 1):
        j = (j + 1) % n
        print(f"  bước {step}: tới vị trí {j} ({s[j]!r})")
    return s[j]


assert trace_one_char_steps("dart", 3, 1) == "d"

ký tự thứ 1 ('a') đi 3 bước:
  bước 1: tới vị trí 2 ('r')
  bước 2: tới vị trí 3 ('t')
  bước 3: tới vị trí 0 ('d')


## 9. Code nộp LeetCode

Bản nộp dùng **Cách 3** (xoay chuỗi bằng cắt lát): ngắn nhất, rõ ý nghĩa "dịch vòng", `O(n)`.

In [21]:
class Solution:
    def getEncryptedString(self, s: str, k: int) -> str:
        r = k % len(s)           # k có thể lớn hơn n: chỉ cần phần dư
        return s[r:] + s[:r]     # xoay trái r vị trí: ký tự i được thay bằng s[(i + k) % n]


sol = Solution()
for s, k, expected in examples:
    print(repr(s), k, "->", repr(sol.getEncryptedString(s, k)), "(kỳ vọng", repr(expected), ")")

'dart' 3 -> 'tdar' (kỳ vọng 'tdar' )
'aaa' 1 -> 'aaa' (kỳ vọng 'aaa' )


## 10. Vì sao thuật toán đúng

- **Công thức vị trí.** Ký tự đứng sau `s[i]` đúng `k` bước (theo vòng) nằm ở `(i + k) % n`: mỗi bước cộng `1` và mỗi lần vượt `n - 1` thì quay về `0`, tức lấy dư theo `n`.
- **`k` có thể rút về `r = k % n`.** `(i + k) % n = (i + (k % n)) % n` vì cộng thêm một bội của `n` không đổi số dư.
- **Chính là phép xoay trái.** Với `0 <= r < n`: nếu `i < n - r` thì `(i + r) % n = i + r` nên ký tự ở vị trí `i` của kết quả là `s[i + r]`, tức các ký tự của `s[r:]` theo thứ tự (độ dài `n - r`); nếu `i >= n - r` thì `(i + r) % n = i + r - n` nên là `s[i + r - n]`, tức các ký tự của `s[:r]` theo thứ tự. Ghép lại: `s[r:] + s[:r]`.
- **Mọi ký tự được xử lý độc lập từ `s` gốc.** Mỗi ký tự mới đọc từ **chuỗi gốc** (không phải từ chuỗi đã sửa dở) nên các phép thay thế không ảnh hưởng nhau.
- **Trường hợp `r = 0`:** `s[0:] + s[:0] = s`, đúng vì đi một bội của `n` bước về chỗ cũ.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` và `k` là bước dịch (`n <= 100`, `k <= 10^4`).

| Cách | Time | Space |
|---|---|---|
| 1. Bước từng ký tự `k` lần | `O(n · k)` | `O(n)` |
| 2. Nhảy thẳng `(i + k) % n` | `O(n)` | `O(n)` |
| 3. Cắt lát `s[r:] + s[:r]` | `O(n)` | `O(n)` |
| 4. Nhân đôi `(s + s)[r:r + n]` | `O(n)` | `O(n)` |
| 5. `deque.rotate(-k)` | `O(n)` | `O(n)` |

Cách 1 tối đa `10^6` bước (vẫn qua trong Python nhưng chậm hơn hẳn); các cách còn lại chỉ `~100` bước. Không gian `O(n)` là bắt buộc vì phải tạo chuỗi kết quả.

## 12. Bộ test

Ta gom mọi cách vào `impls` rồi so với kỳ vọng.

In [22]:
impls = {
    "step_by_step": encrypt_step_by_step,
    "jump": encrypt_jump,
    "slice": encrypt_slice,
    "doubled": encrypt_doubled,
    "deque": encrypt_deque,
    "leetcode": Solution().getEncryptedString,
    "leetcode_c2": SolutionJump().getEncryptedString,
    "leetcode_c3": SolutionSlice().getEncryptedString,
    "leetcode_c4": SolutionDoubled().getEncryptedString,
    "leetcode_c5": SolutionDeque().getEncryptedString,
}


def check_all(s, k, expected, label):
    for key, fn in impls.items():
        actual = fn(s, k)
        assert actual == expected, (label, key, s, k, actual, expected)
    print("PASS", label)


for idx, (s, k, expected) in enumerate(examples, 1):
    check_all(s, k, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [23]:
check_all("a", 1, "a", "một ký tự, k = 1")
check_all("a", 10000, "a", "một ký tự, k lớn")
check_all("ab", 1, "ba", "n = 2, k = 1: đảo")
check_all("ab", 2, "ab", "k = n: về chỗ cũ")
check_all("abc", 3, "abc", "k = n")
check_all("abc", 4, "bca", "k = n + 1 (k > n)")
check_all("abc", 10000, "bca", "k = 10^4, 10000 % 3 = 1")
check_all("abcde", 1, "bcdea", "k = 1")
check_all("abcde", 4, "eabcd", "k = n - 1")
check_all("hello", 2, "llohe", "bài tính tay trong mục 17")
alpha = string.ascii_lowercase
check_all(alpha, 26, alpha, "26 ký tự, k = 26")
check_all(alpha, 1, alpha[1:] + alpha[:1], "26 ký tự, k = 1")
check_all("a" * 100, 9999, "a" * 100, "100 ký tự giống nhau")
big = "".join(chr(97 + i % 26) for i in range(100))
check_all(big, 10000, big, "n = 100, k = 10^4 chia hết 100 nên giữ nguyên")
check_all(big, 9999, big[99:] + big[:99], "n = 100, k = 9999: 9999 % 100 = 99")

PASS một ký tự, k = 1
PASS một ký tự, k lớn
PASS n = 2, k = 1: đảo
PASS k = n: về chỗ cũ
PASS k = n
PASS k = n + 1 (k > n)
PASS k = 10^4, 10000 % 3 = 1
PASS k = 1
PASS k = n - 1
PASS bài tính tay trong mục 17
PASS 26 ký tự, k = 26
PASS 26 ký tự, k = 1
PASS 100 ký tự giống nhau
PASS n = 100, k = 10^4 chia hết 100 nên giữ nguyên


PASS n = 100, k = 9999: 9999 % 100 = 99


### 12.2. Phản ví dụ cho các cách làm sai

Năm lỗi phổ biến (chi tiết ở mục 13): (a) không `% n`; (b) sửa tại chỗ rồi đọc lại; (c) đi sai chiều; (d) nhầm với dịch bảng chữ cái (Caesar); (e) cắt `s[k:] + s[:k]` khi `k > n`.

In [24]:
def wrong_no_mod(s, k):
    n = len(s)
    return "".join(s[i + k] for i in range(n))                           # SAI: IndexError


def wrong_inplace(s, k):
    n = len(s)
    cs = list(s)
    for i in range(n):
        cs[i] = cs[(i + k) % n]                                          # SAI: đọc từ chuỗi đã bị sửa
    return "".join(cs)


def wrong_direction(s, k):
    n = len(s)
    return "".join(s[(i - k) % n] for i in range(n))                     # SAI: lùi thay vì tiến


def wrong_caesar(s, k):
    return "".join(chr((ord(c) - 97 + k) % 26 + 97) for c in s)          # SAI: dịch bảng chữ cái


def wrong_slice_no_mod(s, k):
    return s[k:] + s[:k]                                                  # SAI khi k > n


try:
    wrong_no_mod("dart", 3)
    raise AssertionError("đáng lẽ phải lỗi")
except IndexError as e:
    print("wrong_no_mod: IndexError:", e)
print(wrong_inplace("dart", 1), "!=", Solution().getEncryptedString("dart", 1))
print(wrong_direction("abcde", 1), "!=", Solution().getEncryptedString("abcde", 1))
print(wrong_caesar("dart", 3), "!=", Solution().getEncryptedString("dart", 3))
print(wrong_slice_no_mod("abc", 4), "!=", Solution().getEncryptedString("abc", 4))
assert wrong_inplace("dart", 1) != "artd"
assert wrong_direction("abcde", 1) != "bcdea"
assert wrong_caesar("dart", 3) != "tdar"
assert wrong_slice_no_mod("abc", 4) != "bca"

wrong_no_mod: IndexError: string index out of range
arta != artd
eabcd != bcdea
gduw != tdar
abc != bca


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Đầu vào hợp lệ: `s` gồm chữ cái thường độ dài `1..100`, `k` từ `1` đến `10^4`. Ta kiểm tra: (1) **ngẫu nhiên** `2000` cặp `(s, k)` với `k` nhỏ để so cả Cách 1; (2) `300` cặp với `k` lớn (không dùng Cách 1 do chậm); (3) **vét cạn** mọi `n` từ `1` đến `8` và mọi `k` từ `1` đến `40` trên chuỗi `"abcdefgh"[:n]`; (4) tính chất: mã hóa với `k` rồi với `n - k % n` trả về chuỗi gốc.

In [25]:
rng = random.Random(3210)
for trial in range(2000):
    n_len = rng.randint(1, 100)
    s = "".join(rng.choice(string.ascii_lowercase) for _ in range(n_len))
    k = rng.randint(1, 300)
    ref = encrypt_step_by_step(s, k)
    for key, fn in impls.items():
        assert fn(s, k) == ref, (key, s, k)
print("PASS 2000 cặp (s, k) ngẫu nhiên có k <= 300, so cả cách từng bước")

fast_impls = {key: fn for key, fn in impls.items() if key != "step_by_step"}
for trial in range(300):
    n_len = rng.randint(1, 100)
    s = "".join(rng.choice(string.ascii_lowercase) for _ in range(n_len))
    k = rng.randint(1, 10 ** 4)
    ref = encrypt_slice(s, k)
    assert len(ref) == len(s) and sorted(ref) == sorted(s)
    for key, fn in fast_impls.items():
        assert fn(s, k) == ref, (key, s, k)
print("PASS 300 cặp (s, k) với k lên tới 10^4")

total = 0
for n_len in range(1, 9):
    s = "abcdefgh"[:n_len]
    for k in range(1, 41):
        ref = encrypt_step_by_step(s, k)
        for key, fn in impls.items():
            assert fn(s, k) == ref
        total += 1
print("PASS vét cạn", total, "cặp (n, k) trên chuỗi các chữ khác nhau")

for trial in range(500):
    n_len = rng.randint(1, 50)
    s = "".join(rng.choice(string.ascii_lowercase) for _ in range(n_len))
    k = rng.randint(1, 10 ** 4)
    enc = Solution().getEncryptedString(s, k)
    back = Solution().getEncryptedString(enc, n_len - k % n_len)
    assert back == s
print("PASS tính chất: mã hóa k rồi mã hóa n - k % n trả về chuỗi gốc")

PASS 2000 cặp (s, k) ngẫu nhiên có k <= 300, so cả cách từng bước
PASS 300 cặp (s, k) với k lên tới 10^4
PASS vét cạn 320 cặp (n, k) trên chuỗi các chữ khác nhau
PASS tính chất: mã hóa k rồi mã hóa n - k % n trả về chuỗi gốc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đọc `s[i + k]` mà không có `% n`

In [26]:
def sai_khong_mod(s, k):
    n = len(s)
    return "".join(s[i + k] for i in range(n))      # SAI


try:
    sai_khong_mod("dart", 3)
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().getEncryptedString("dart", 3))

IndexError: string index out of range
đúng: tdar


**Vì sao sai:** với `n = 4`, `k = 3`, `i = 1` ta đọc `s[4]` — vượt chỉ số hợp lệ lớn nhất `3`. Đề yêu cầu **đi vòng** nên khi hết chuỗi phải quay về đầu. **Cách sửa:** `s[(i + k) % n]`.

### Lỗi 2: Sửa tại chỗ rồi đọc lại phần đã sửa

In [27]:
def sai_sua_tai_cho(s, k):
    n = len(s)
    cs = list(s)
    for i in range(n):
        cs[i] = cs[(i + k) % n]          # SAI: cs[(i+k)%n] có thể đã bị ghi đè ở vòng trước
    return "".join(cs)


print("sai:", sai_sua_tai_cho("dart", 1), "| đúng:", Solution().getEncryptedString("dart", 1))
print("sai:", sai_sua_tai_cho("abcde", 2), "| đúng:", Solution().getEncryptedString("abcde", 2))

sai: arta | đúng: artd
sai: cdecd | đúng: cdeab


**Vì sao sai:** ở `s = "dart"`, `k = 1`: bước `i = 0` đổi `cs[0]` thành `'a'`, ... đến `i = 3` ta cần ký tự gốc `s[0] = 'd'` nhưng `cs[0]` đã là `'a'`. Đề thay thế **cùng lúc** từ chuỗi gốc. **Cách sửa:** luôn đọc từ `s` (chuỗi gốc), ghi vào list `cs` riêng (hoặc dùng cắt lát).

### Lỗi 3: Đi sai chiều (lùi thay vì tiến)

In [28]:
def sai_di_lui(s, k):
    n = len(s)
    return "".join(s[(i - k) % n] for i in range(n))      # SAI: i - k


print("sai:", sai_di_lui("abcde", 1), "| đúng:", Solution().getEncryptedString("abcde", 1))

sai: eabcd | đúng: bcdea


**Vì sao sai:** "ký tự đứng **sau** `c`" nghĩa là tiến về phía chỉ số **lớn hơn** (`i + k`). `i - k` cho chuỗi xoay **phải** (`"eabcd"`). Hãy thử thêm `"abcde"`, `k = 1` để thấy rõ hai chiều cho kết quả khác nhau (`"bcdea"` so với `"eabcd"`). **Cách sửa:** `(i + k) % n`.

### Lỗi 4: Nhầm "sau `k` vị trí trong chuỗi" với "dịch `k` chữ cái trong bảng chữ cái"

In [29]:
def sai_caesar(s, k):
    return "".join(chr((ord(c) - 97 + k) % 26 + 97) for c in s)      # SAI


print("sai:", sai_caesar("dart", 3), "| đúng:", Solution().getEncryptedString("dart", 3))

sai: gduw | đúng: tdar


**Vì sao sai:** đề thay `c` bằng **ký tự cách nó `k` vị trí trong chính chuỗi `s`**, chứ không phải chữ cái cách `k` chữ trong bảng `a..z` (như mã Caesar). Với `"dart"`, `k = 3` đúng là `"tdar"` (các chữ vẫn là `d, a, r, t`), còn Caesar cho chữ hoàn toàn khác. **Cách sửa:** làm việc với **chỉ số trong `s`**, không dùng `ord`.

### Lỗi 5: `s[k:] + s[:k]` khi `k > n`

In [30]:
def sai_cat_khong_mod(s, k):
    return s[k:] + s[:k]          # SAI khi k > n


print("sai:", sai_cat_khong_mod("abc", 4), "| đúng:", Solution().getEncryptedString("abc", 4))
print("sai:", sai_cat_khong_mod("abc", 10000), "| đúng:", Solution().getEncryptedString("abc", 10000))

sai: abc | đúng: bca
sai: abc | đúng: bca


**Vì sao sai:** khi `k >= n`, `s[k:]` là chuỗi rỗng và `s[:k]` là cả chuỗi nên kết quả luôn là `s`. Ràng buộc cho phép `k` lên tới `10^4` trong khi `n <= 100`, nên chuyện này xảy ra thường xuyên. **Cách sửa:** `r = k % n` trước khi cắt.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thay mỗi phần tử bằng phần tử cách nó `k` bước **theo vòng**" ⇒ `(i + k) % n`.
- Nếu **mọi** phần tử dịch cùng một khoảng thì đó là phép **xoay**: `s[r:] + s[:r]` với `r = k % n`.
- `k` lớn hơn `n` ⇒ luôn rút `k % n` trước khi cắt lát (cắt lát không tự "quấn vòng", còn `deque.rotate` thì có).
- Muốn xoay **phải** `r`: xoay trái `t = (-r) % n`, tức `s[t:] + s[:t]`.
- Không sửa chuỗi/list **tại chỗ** nếu còn cần đọc giá trị cũ; đọc từ bản gốc, ghi vào bản mới.
- Chuỗi Python bất biến: tạo list rồi `"".join(...)` hoặc dùng cắt lát/biểu thức sinh.
- Test `k = n`, `k = n + 1`, `k = 10^4` và `n = 1` trước khi nộp.

## 15. Biến thể của bài

1. **Giải mã:** cho chuỗi đã mã hóa và `k`, khôi phục `s` (xoay phải `k % n`).
2. **`k` khác nhau cho từng vị trí:** ký tự `i` thay bằng `s[(i + k[i]) % n]`.
3. **Số `k` nhỏ nhất** (`k > 0`) để mã hóa trả về đúng chuỗi gốc.
4. **Mã hóa nhiều lần:** áp dụng `m` lần liên tiếp (tương đương `k · m`).
5. Bài liên quan: **Rotate Array (189)**, **Rotate String (796)**, **Rotate List (61)**, **Shifting Letters (848)**.

In [31]:
def decrypt(t, k):
    n = len(t)
    r = (-k) % n                  # xoay phải k = xoay trái (n - k % n) % n
    return t[r:] + t[:r]


def encrypt_each(s, ks):
    n = len(s)
    return "".join(s[(i + ks[i]) % n] for i in range(n))


def smallest_k_same(s):
    n = len(s)
    for k in range(1, n + 1):
        if Solution().getEncryptedString(s, k) == s:
            return k


def encrypt_m_times(s, k, m):
    result = s
    for _ in range(m):
        result = Solution().getEncryptedString(result, k)
    return result


print(decrypt("tdar", 3))                       # dart
print(encrypt_each("dart", [0, 1, 2, 3]))        # drdr: ký tự i thay bằng s[(i + i) % 4]
print(smallest_k_same("abab"), smallest_k_same("aaaa"), smallest_k_same("abc"))      # 2 1 3
print(encrypt_m_times("dart", 3, 2), Solution().getEncryptedString("dart", 6))        # trùng nhau

dart
drdr
2 1 3
rtda rtda


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bước từng ký tự `k` lần | nhích `j = (j + 1) % n` `k` lần | `O(n·k)` | `O(n)` | hiểu đề, kiểm chứng |
| 2. Nhảy thẳng `(i + k) % n` | công thức vị trí | `O(n)` | `O(n)` | bám đề, không cần nghĩ tới xoay |
| 3. Cắt lát | xoay trái `r = k % n` | `O(n)` | `O(n)` | **nên nộp**, ngắn nhất |
| 4. Nhân đôi chuỗi | `(s + s)[r:r + n]` | `O(n)` | `O(n)` | khi quen với "chuỗi vòng" |
| 5. `deque.rotate` | xoay trái có sẵn | `O(n)` | `O(n)` | khi cần xoay nhiều lần |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"dart"[3]`, `7 % 4`, `"dart"[1:] + "dart"[:1]`, `10000 % 3`, `Solution().getEncryptedString("abc", 1)`.

**Bài 2 (tính tay).** Với `s = "hello"`, `k = 2` (`n = 5`): lập bảng `i`, `s[i]`, `(i + k) % n`, ký tự mới và cho biết chuỗi mã hóa.

**Bài 3 (điền TODO).** Hoàn thành `encrypt_todo(s, k)` theo Cách 2 (nhảy thẳng).

In [32]:
def encrypt_todo(s, k):
    # TODO 1: n = len(s); cs = []
    # TODO 2: for i in range(n): thêm vào cs ký tự s[(i + k) % n]
    # TODO 3: trả về "".join(cs)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `decrypt_todo(t, k)` là **phép ngược** của mã hóa: với mọi `s`, `k` thì `decrypt(encrypt(s, k), k) == s`. Gợi ý: xoay ngược chiều.

**Bài 5 (biến thể / mini challenge).** (a) Viết `encrypt_each_todo(s, ks)` trong đó ký tự `i` được thay bằng ký tự cách nó `ks[i]` bước (mỗi vị trí một `k` riêng). (b) Viết `smallest_k_same_todo(s)` trả về `k > 0` nhỏ nhất sao cho mã hóa `s` với `k` cho ra đúng `s`; dự đoán kết quả cho `"abab"`, `"aaaa"`, `"abc"`.

In [33]:
def decrypt_todo(t, k):
    # TODO: xoay ngược chiều (xoay phải k % n), hoặc xoay trái (n - k % n)
    raise NotImplementedError


def encrypt_each_todo(s, ks):
    # TODO: ký tự i -> s[(i + ks[i]) % n]
    raise NotImplementedError


def smallest_k_same_todo(s):
    # TODO: thử k = 1..n, trả về k đầu tiên mà mã hóa không đổi chuỗi
    raise NotImplementedError

# KHÔNG gọi các hàm TODO này khi Run All.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `'t'`, `3`, `'artd'`, `1`, `'bca'`. Bài 3: ký tự mới là `s[(i + k) % n]`.
- **Mức 2 (vừa):** Bài 2: `i=0 -> 2 (l)`, `i=1 -> 3 (l)`, `i=2 -> 4 (o)`, `i=3 -> 0 (h)`, `i=4 -> 1 (e)` ⇒ `"llohe"`. Bài 4: mã hóa là xoay trái `r = k % n`; giải mã là xoay **phải** `r` tức xoay trái `(-k) % n`.
- **Mức 3 (sâu):** Bài 5(a): chỉ khác Cách 2 ở chỗ dùng `ks[i]` thay `k`. Bài 5(b): `"abab"` có chu kỳ `2` nên `k = 2`; `"aaaa"` cho `k = 1`; `"abc"` không có chu kỳ ngắn hơn `n` nên `k = 3`. Chỉ cần thử `k = 1..n` (vì `k = n` luôn trả về `s`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert "dart"[3] == "t"
assert 7 % 4 == 3
assert "dart"[1:] + "dart"[:1] == "artd"
assert 10000 % 3 == 1
assert Solution().getEncryptedString("abc", 1) == "bca"

# Bài 2
table = [((i + 2) % 5, "hello"[(i + 2) % 5]) for i in range(5)]
assert table == [(2, "l"), (3, "l"), (4, "o"), (0, "h"), (1, "e")]
assert Solution().getEncryptedString("hello", 2) == "llohe"


# Bài 3
def encrypt_todo_ans(s, k):
    n = len(s)
    cs = []
    for i in range(n):
        cs.append(s[(i + k) % n])
    return "".join(cs)


for s, k, expected in examples:
    assert encrypt_todo_ans(s, k) == expected


# Bài 4
def decrypt_todo_ans(t, k):
    r = (-k) % len(t)
    return t[r:] + t[:r]


rng = random.Random(5)
for _ in range(500):
    s = "".join(rng.choice(string.ascii_lowercase) for _ in range(rng.randint(1, 30)))
    k = rng.randint(1, 10 ** 4)
    assert decrypt_todo_ans(Solution().getEncryptedString(s, k), k) == s


# Bài 5
def encrypt_each_todo_ans(s, ks):
    n = len(s)
    return "".join(s[(i + ks[i]) % n] for i in range(n))


def smallest_k_same_todo_ans(s):
    for k in range(1, len(s) + 1):
        if Solution().getEncryptedString(s, k) == s:
            return k


assert encrypt_each_todo_ans("dart", [3, 3, 3, 3]) == "tdar"
assert encrypt_each_todo_ans("dart", [0, 0, 0, 0]) == "dart"
assert encrypt_each_todo_ans("abcd", [1, 2, 3, 0]) == "bdbd"
assert smallest_k_same_todo_ans("abab") == 2
assert smallest_k_same_todo_ans("aaaa") == 1
assert smallest_k_same_todo_ans("abc") == 3
print("Tất cả đáp án bài tập đều PASS")

Tất cả đáp án bài tập đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mã hóa = xoay trái `k % n`)

```python
class Solution:
    def getEncryptedString(self, s: str, k: int) -> str:
        r = k % len(s)
        return s[r:] + s[:r]
```

Hoặc từng ký tự: `"".join(s[(i + k) % n] for i in range(n))`.

**Checklist**

- [ ] Vị trí mới là `(i + k) % n` (cộng, không trừ) và đã `% n`?
- [ ] Đọc ký tự từ chuỗi **gốc** `s` (không sửa tại chỗ rồi đọc lại)?
- [ ] Nếu cắt lát, đã đặt `r = k % n` trước (vì `k` có thể lớn hơn `n`)?
- [ ] Không nhầm với dịch bảng chữ cái kiểu Caesar?
- [ ] Đã thử `"dart", 3` (`"tdar"`), `"aaa", 1`, `n = 1`, `k = n`, `k = 10^4`?